# XGBoost By-Gene mRNA Data Investigation

This notebook uses the strongest realistic XGBoost setup from the saved baseline notebook: by-gene evaluation, mRNA included, and frozen Optuna-tuned hyperparameters.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Build Current Pipeline Data

This investigation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`.

Important note: the frozen XGBoost hyperparameters were tuned in the earlier baseline notebook, not re-tuned on this stricter pipeline. That is acceptable for data investigation, but the results should be treated as investigation-first rather than final model selection.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)
X, groups, y = pipeline.prepare_for_classical_ml(
    enriched_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)

mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = enriched_df.loc[mask].reset_index(drop=True).copy()

analysis_df["patent_group"] = analysis_df.get(
    "patent_ID", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
analysis_df["Authorization_status"] = analysis_df.get(
    "Authorization_status", pd.Series(index=analysis_df.index, dtype=object)
).fillna("UNKNOWN")
analysis_df["Title"] = analysis_df.get(
    "Title", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")

print("Analysis dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## Frozen Hyperparameters

These are the saved Optuna hyperparameters from the earlier by-gene + mRNA XGBoost notebook.

In [3]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## By-Gene Out-of-Fold Evaluation

This is the harder and more biologically realistic view. Each prediction comes from a fold where that gene group was held out, so weak slices here are stronger candidates for transfer-poisoning before deep learning.

In [4]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=30, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,24133,11311,54,16
1,2,24133,11311,54,16
2,3,24136,11308,54,16


## Fold Summary

## Overall Metrics

In [5]:
metrics_df

,spearman,pearson,rmse,mae
0,0.45886,0.453422,31.421689,24.568816


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations.

In [6]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,0.00030,40,-0.195440,13.421261,5.699000,0.280597
1,0.01600,114,-0.101409,21.098231,21.496842,26.864985
2,2.22220,48,-0.039251,12.532996,78.291042,75.263405
3,0.00064,114,-0.030506,11.172622,-1.481053,3.362773
4,100.00000,1036,-0.024001,30.540463,51.639293,54.521019
5,0.30000,147,-0.021149,22.393722,43.333197,53.068233
6,0.00100,66,-0.000564,16.243110,7.866212,-0.251111
7,0.00300,64,0.003869,15.036281,3.641250,2.310574
8,0.01000,197,0.004840,24.999101,36.776497,21.073807
9,150.00000,87,0.004981,21.975498,37.046782,43.936913


## Concentration + Gene Hotspots

In [7]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,20.00000,APP,168,-0.556349,38.583055,41.410714,57.806583
1,0.00064,HSD17B13,13,-0.533333,17.031856,11.537692,0.787329
2,100.00000,INHBE,180,-0.440447,37.574473,62.262167,39.903522
3,0.00100,AGT,20,-0.375189,12.456005,-0.747500,3.952451
4,0.24700,HSD17B13,16,-0.304636,33.647057,78.275000,50.828087
5,0.00030,AGT,19,-0.244074,12.303843,7.070526,2.115313
6,2.22200,HSD17B13,16,-0.226471,27.278895,87.950000,66.213806
7,100.00000,PCSK9,293,-0.212300,31.238838,54.427304,54.176559
8,0.00030,HSD17B13,21,-0.211826,14.432258,4.458095,-1.379385
9,0.08200,HSD17B13,15,-0.207143,34.241547,62.420000,36.845055


## Concentration + Patent Hotspots

In [16]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,20.0000,US20220380773A1,168,-0.556349,38.583055,41.410714,57.806583
1,10.0000,CN112313335A,180,-0.506693,53.909804,92.920111,39.010307
2,100.0000,WO2023003922A1,180,-0.440447,37.574473,62.262167,39.903522
3,0.1000,CN112313335A,182,-0.401716,84.790607,98.662308,13.871700
4,0.1000,WO2022089486A1,20,-0.394430,49.222262,62.100000,12.877737
5,0.0010,WO2023088427A1,20,-0.375189,12.456005,-0.747500,3.952451
6,0.2470,CN116940681A,16,-0.304636,33.647057,78.275000,50.828087
7,0.0003,CN116940681A,16,-0.270987,12.980646,5.756250,-0.935589
8,0.1000,CN116515835A,163,-0.260603,20.166744,63.156135,60.760334
9,0.0003,WO2023088427A1,19,-0.244074,12.303843,7.070526,2.115313


## Experimental Feature Issues

These summaries highlight which experimental contexts are associated with poor ranking or high error in this XGBoost setup.

In [9]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Primary mouse hepatocytes,306,58.485055,70.103295,-0.176535,23.606536,44.061111
1,Non-human hepatocytes,189,36.277806,42.050668,0.094603,22.585185,58.668114
2,Primary human hepatocytes,2925,23.720921,29.400667,0.228048,50.909699,52.130947
3,Primary Cynomolgus Monkey Hepatocytes,4229,26.533184,33.501733,0.290038,34.587378,33.581558
4,Hela,1884,26.905052,32.929938,0.360365,41.483854,38.581261
5,HepG2,1623,21.659766,28.560982,0.373450,40.655588,47.879112
6,Hep3B,11837,28.088053,35.189296,0.380525,42.291176,36.505539
7,Human iPSC-derived cortical neurons,196,23.872235,29.105819,0.402574,42.401531,47.698009
8,Huh7,1546,21.568773,27.134123,0.513225,40.809056,42.136162
9,HEK293 Cells,139,19.880132,24.425665,0.530534,69.970051,61.189964


## Concentration Issue Summary

In [10]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,0.00030,40,13.421261,15.514410,-0.195440,5.699000,0.280597
1,0.01600,114,21.098231,25.874606,-0.101409,21.496842,26.864985
2,2.22220,48,12.532996,16.579036,-0.039251,78.291042,75.263405
3,0.00064,114,11.172622,14.514477,-0.030506,-1.481053,3.362773
4,100.00000,1036,30.540463,37.388962,-0.024001,51.639293,54.521019
5,0.30000,147,22.393722,26.700970,-0.021149,43.333197,53.068233
6,0.00100,66,16.243110,20.661664,-0.000564,7.866212,-0.251111
7,0.00300,64,15.036281,18.562715,0.003869,3.641250,2.310574
8,0.01000,197,24.999101,32.521327,0.004840,36.776497,21.073807
9,150.00000,87,21.975498,27.310116,0.004981,37.046782,43.936913


## Time Issue Summary

In [11]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,40.0,194,15.966733,20.487457,0.222517,27.283505,27.174477
1,24.0,23975,26.642361,33.760808,0.399411,41.702802,39.082073
2,168.0,196,23.872235,29.105819,0.402574,42.401531,47.698009
3,72.0,1391,20.378487,24.884504,0.436481,15.703084,28.467026
4,48.0,8158,19.430126,24.905141,0.560985,48.721842,47.656406


## Patent Issue Summary

In [12]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,CN112313335A,362,69.435511,72.131256,-0.771602,95.807072,26.371561
1,US20220380773A1,168,38.583055,44.331993,-0.556349,41.410714,57.806583
2,WO2023213284A1,48,28.629046,31.765363,-0.382390,86.482083,59.822765
3,CN109957567B,39,29.938579,35.196310,-0.151795,63.641026,48.469551
4,CN113980966A,57,55.215853,57.839635,-0.146120,83.438596,28.222744
5,CN117210468A,222,19.640954,25.188394,-0.025943,25.166667,30.759436
6,WO2022121959A1,97,40.448960,47.671961,-0.004715,64.775773,45.962517
7,CN117448322A,131,47.470579,53.833370,0.032327,45.774809,9.441285
8,TW202321444A,189,36.277806,42.050668,0.094603,22.585185,58.668114
9,WO2023091644A2,1437,23.567018,29.235842,0.113417,48.167627,51.295597


## Authorization Issue Summary

In [13]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,222,19.640954,25.188394,-0.025943,25.166667,30.759436
1,Substantive Examination,15965,27.093010,33.817959,0.467493,46.288366,35.196400
2,Unknown Status,12626,23.640399,30.798977,0.467742,33.490578,43.205025
3,Withdrawn,1082,23.577476,28.934047,0.473493,30.183919,32.938908
4,Granted,1702,24.320491,30.194381,0.481170,49.467368,46.113258
5,UNKNOWN,2333,13.429809,16.899477,0.623746,63.832783,65.178070


## Prediction Preview

In [14]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,LPA,Hep3B,1.0,24.0,CN108368506A,-1.2,2.115234,-3.315234,3.315234
1,LPA,Hep3B,1.0,24.0,CN108368506A,22.4,17.298868,5.101132,5.101132
2,LPA,Hep3B,1.0,24.0,CN108368506A,29.2,32.004311,-2.804311,2.804311
3,LPA,Hep3B,1.0,24.0,CN108368506A,55.9,36.135521,19.764479,19.764479
4,LPA,Hep3B,1.0,24.0,CN108368506A,75.8,40.460178,35.339822,35.339822
5,LPA,Hep3B,1.0,24.0,CN108368506A,83.4,53.910782,29.489218,29.489218
6,LPA,Hep3B,1.0,24.0,CN108368506A,29.8,6.716002,23.083998,23.083998
7,LPA,Hep3B,1.0,24.0,CN108368506A,72.8,29.288115,43.511885,43.511885
8,LPA,Hep3B,1.0,24.0,CN108368506A,71.0,40.356281,30.643719,30.643719
9,LPA,Hep3B,1.0,24.0,CN108368506A,17.5,32.756493,-15.256493,15.256493


## Interpretation Notes

- This is the more realistic notebook for your project because it evaluates transfer to unseen genes. Overall performance drops a lot relative to the random-split view: Spearman is about `0.459`, Pearson about `0.453`, RMSE about `31.42`, and MAE about `24.57`.
- The big gap between random split and by-gene split means a lot of the challenge is not simple fitting, but generalization across targets. That is exactly the regime that matters for deep learning.
- Concentration-only behavior is much weaker here. Several concentrations have near-zero or negative Spearman, especially ultra-low-dose regimes such as `0.00030`, `0.00064`, `0.001`, `0.003`, and `0.010 nM`, but also some larger-sample regimes like `100 nM` and `20 nM`. This means concentration effects are interacting with target or source context rather than acting as a simple global rule.
- The `Concentration + Gene` view is much more informative than concentration alone. The worst slices include `APP @ 20 nM`, `INHBE @ 100 nM`, `PCSK9 @ 100 nM`, and many `HSD17B13` combinations across several concentrations. This suggests that some of the hardest regimes are target-specific rather than just low-dose globally.
- The `Concentration + Patent` view is even more suspicious. The clearest hotspots include `CN112313335A @ 10.0 nM`, `CN112313335A @ 0.1 nM`, `US20220380773A1 @ 20.0 nM`, `WO2023003922A1 @ 100 nM`, `US20220117999A1 @ 10.0 nM`, and `WO2022121959A1` slices. These look like strong source/protocol heterogeneity candidates.
- Cell-type behavior is also much worse in the by-gene view. `Primary mouse hepatocytes` stands out again as a serious warning sign with negative Spearman and very high error. `Non-human hepatocytes` is also weak. Broader contexts like `Primary human hepatocytes`, `Primary Cynomolgus Monkey Hepatocytes`, and `Hep3B` are not catastrophic, but they are clearly harder than they looked in the random split.
- Time-of-administration does not look like the main poisoning axis here. `72h` is not among the worst signals in this notebook, so XGBoost does not support broad deletion of all `72h` rows.
- Authorization status is also not the main issue. The weak behavior there seems secondary to patent/source composition rather than a clean standalone filtering rule.
- Relative to the RF notebooks, there is partial overlap rather than perfect overlap. `Primary mouse hepatocytes` and `US20220117999A1` remain suspicious across models. The strongest XGBoost-specific warning signal is the broader patent hotspot pattern, especially around `CN112313335A`, `US20220380773A1`, and `WO2022121959A1`.

## Conclusions And Next Step

- This notebook suggests the main risk before deep learning is not broad global poisoning, but target-specific and patent-specific transfer failure.
- The cleanest next step is **not** to remove whole concentrations globally. Instead, prioritize targeted review or ablation of overlapping weak slices across RF and XGBoost.
- The strongest cross-notebook candidates to inspect next are:
  - `Primary mouse hepatocytes`
  - `US20220117999A1` hotspot slices
  - `WO2022121959A1` hotspot slices
  - `US20220380773A1 @ 20 nM`
  - `CN112313335A` hotspot slices
- I would also manually review the repeated `HSD17B13` and `APP @ 20 nM` failures before deciding whether they are poisoning or just difficult biology.
- The best practical next step is to build one overlap table across RF and XGBoost suspicious subsets, then test only the shared worst hotspots as targeted ablations before CNN training.
